In [1]:
# ======================================================================================
# 1. Install required libraries for Colab and YouTube integration
# ======================================================================================
!pip install yt-dlp pafy youtube-dl
!pip install opencv-python matplotlib tensorflow
!apt-get install unrar

import os
import cv2
import math
import random
import numpy as np
import datetime as dt
import tensorflow as tf
from collections import deque
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow # Replacement for cv2.imshow in Colab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 83.7 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [6]:
# ======================================================================================
# 2. Download UCF11 Dataset and Extract
# ======================================================================================
print("Downloading dataset...")
!wget -nc --no-check-certificate https://www.crcv.ucf.edu/data/UCF11_updated_mpg.rar
!unrar x UCF11_updated_mpg.rar -inul -y
print("Dataset downloaded and extracted successfully! ")

# ======================================================================================
# 3. Global Settings and Class Selection
# ======================================================================================
DATASET_DIR = "UCF11_updated_mpg"
CLASSES_LIST = ["basketball", "biking", "diving"]

IMAGE_HEIGHT, IMAGE_WIDTH = 64, 64
SEQUENCE_LENGTH = 20 # Number of frames per video sequence

File ‘UCF11_updated_mpg.rar’ already there; not retrieving.

Dataset downloaded and extracted successfully! 


In [7]:
# ======================================================================================
# 4. Function for Frame Extraction and Preprocessing
# ======================================================================================
def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))

    # Calculate the interval to skip frames and get a fixed sequence length
    skip_frames_window = max(int(video_frames_count/SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break
        # Resize and normalize the frame
        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        normalized_frame = resized_frame / 255
        frames_list.append(normalized_frame)

    video_reader.release()
    return frames_list

In [10]:
# ======================================================================================
# 5. Function to Create the Dataset (Features and Labels)
# ======================================================================================
def create_dataset():
    features = []
    labels = []

    for class_index, class_name in enumerate(CLASSES_LIST):
        print(f'Extracting Data for Class: {class_name}...')

        # Path to the class folder
        class_path = os.path.join(DATASET_DIR, class_name)

        # This will look for videos in the main folder AND all sub-folders
        for root, dirs, files in os.walk(class_path):
            for file_name in files:
                if file_name.endswith(('.mpg', '.avi', '.mp4')):
                    video_file_path = os.path.join(root, file_name)

                    frames = frames_extraction(video_file_path)

                    # Ensure we only add videos that have the required sequence length
                    if len(frames) == SEQUENCE_LENGTH:
                        features.append(frames)
                        labels.append(class_index)

    features = np.asarray(features)
    labels = np.array(labels)
    return features, labels

# Execute dataset creation
features, labels = create_dataset()

# Check if we actually found data
if len(features) == 0:
    print("Error: No videos found! Please check if DATASET_DIR and CLASSES_LIST are correct.")
else:
    print(f"Success! Total samples found: {len(features)}")
    one_hot_encoded_labels = tf.keras.utils.to_categorical(labels)



# Split the data into Training (75%) and Testing (25%) sets
from sklearn.model_selection import train_test_split
features_train, features_test, labels_train, labels_test = train_test_split(
    features, one_hot_encoded_labels, test_size=0.25, shuffle=True, random_state=42
)


Extracting Data for Class: basketball...
Extracting Data for Class: biking...
Extracting Data for Class: diving...
Success! Total samples found: 439


In [11]:
# ======================================================================================
# 6. Construct the LRCN Model Architecture
# ======================================================================================
def create_lrcn_model():
    model = tf.keras.Sequential([
        # Use TimeDistributed to apply Conv layers to each frame in the sequence
        tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
                                        input_shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3)),
        tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPooling2D((4, 4))),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Dropout(0.25)),

        tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu')),
        tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPooling2D((4, 4))),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Dropout(0.25)),

        tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten()),

        # LSTM layer to handle the temporal (motion) aspect
        tf.keras.layers.LSTM(64),

        # Output layer with Softmax for classification
        tf.keras.layers.Dense(len(CLASSES_LIST), activation='softmax')
    ])

    model.compile(loss='categorical_crossentropy', optimizer='Adam', metrics=["accuracy"])
    return model

model = create_lrcn_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 20, 64, 64, 16) │           448 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 16, 16, 16) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 20, 16, 16, 16) │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 20, 16, 16, 32) │         4,640 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 20, 4, 4, 32)   │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 20, 4, 4, 32)   │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 20, 512)        │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │       147,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 152,995 (597.64 KB)

 Trainable params: 152,995 (597.64 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# ======================================================================================
# 7. Model Training and Saving (Mandatory Step)
# ======================================================================================
print("Starting Training...")
history = model.fit(x=features_train, y=labels_train, epochs=15, batch_size=4,
                    shuffle=True, validation_split=0.2)

student_name = "haneen_alamri"
save_path = f"{student_name}_ucf11_model.h5"
model.save(save_path)

print(f"Model saved successfully as: {save_path}")

Starting Training...
Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 60ms/step - accuracy: 0.5323 - loss: 0.9423 - val_accuracy: 0.7424 - val_loss: 0.6675
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.6426 - loss: 0.6573 - val_accuracy: 0.6515 - val_loss: 0.6990
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.7490 - loss: 0.5591 - val_accuracy: 0.7879 - val_loss: 0.5022
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.8479 - loss: 0.4045 - val_accuracy: 0.8030 - val_loss: 0.4099
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.8517 - loss: 0.3512 - val_accuracy: 0.8182 - val_loss: 0.3918
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.8935 - loss: 0.2750 - val_accuracy: 0.8182 - val_loss: 0.4158
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.9202 - loss: 0.2121 - val_accuracy: 0.8333 - val_loss: 0.4335
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9125 - loss: 0.1818 - val

Model saved successfully as: haneen_alamri_ucf11_model.h5


In [18]:
# ======================================================================================
# YouTube Validation Step
# ======================================================================================
import yt_dlp

def predict_on_youtube_video(video_file_path, output_video_file_path, SEQUENCE_LENGTH):
    video_reader = cv2.VideoCapture(video_file_path)

    # Get video properties
    original_video_width = int(video_reader.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_video_height = int(video_reader.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Preprocess the video to match model input
    frames_list = []
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(video_frames_count/SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success: break
        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        normalized_frame = resized_frame / 255
        frames_list.append(normalized_frame)

    # Make Prediction
    predicted_labels_probabilities = model.predict(np.expand_dims(frames_list, axis=0))[0]
    predicted_label = np.argmax(predicted_labels_probabilities)
    predicted_class_name = CLASSES_LIST[predicted_label]

    print(f'Action Predicted: {predicted_class_name} (Confidence: {predicted_labels_probabilities[predicted_label]:.2f})')

    video_reader.release()
    return predicted_class_name


In [26]:
!yt-dlp -f mp4 "https://www.youtube.com/watch?v=IZ2RvtHiEEU" -o "diving.mp4"

         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[youtube] Extracting URL: https://www.youtube.com/watch?v=IZ2RvtHiEEU
[youtube] IZ2RvtHiEEU: Downloading webpage
[youtube] IZ2RvtHiEEU: Downloading android vr player API JSON
[info] IZ2RvtHiEEU: Downloading 1 format(s): 18
[download] Destination: diving.mp4
[download] 100% of   11.26MiB in 00:00:03 at 3.11MiB/s


In [30]:
!yt-dlp -f mp4 "https://www.youtube.com/shorts/6pxrnZDofKk" -o "Biking.mp4"

         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[youtube] Extracting URL: https://www.youtube.com/shorts/6pxrnZDofKk
[youtube] 6pxrnZDofKk: Downloading webpage
[youtube] 6pxrnZDofKk: Downloading android vr player API JSON
[info] 6pxrnZDofKk: Downloading 1 format(s): 18
[download] Biking.mp4 has already been downloaded
[download] 100% of    5.22MiB


In [28]:
!yt-dlp -f mp4 "https://www.youtube.com/watch?v=8-7JVqPlUJ4" -o "basketball.mp4"

         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[youtube] Extracting URL: https://www.youtube.com/watch?v=8-7JVqPlUJ4
[youtube] 8-7JVqPlUJ4: Downloading webpage
[youtube] 8-7JVqPlUJ4: Downloading android vr player API JSON
[info] 8-7JVqPlUJ4: Downloading 1 format(s): 18
[download] Destination: basketball.mp4
[download] 100% of    4.46MiB in 00:00:00 at 13.47MiB/s


In [38]:
# The list of your manually uploaded videos
manual_test_videos = ['diving.mp4', 'Biking.mp4', 'basketball.mp4']

print("\n--- Starting Final Validation ---")

for i, video_file in enumerate(manual_test_videos):
    print(f"\n--- Testing Video {i+1}: {video_file} ---")

    # Check if the file exists in the Colab sidebar
    if not os.path.exists(video_file):
        print(f" File Error: '{video_file}' not found. Please upload it.")
        continue

    try:
        predicted_class = predict_on_youtube_video(video_file, f'output_{i}.mp4', SEQUENCE_LENGTH)

        # Display the results
        print(f"FINAL PREDICTION: {predicted_class}")

    except Exception as e:
        print(f" Execution Error for {video_file}: {e}")

print("\n--- Validation Process Finished ---")


--- Starting Final Validation ---

--- Testing Video 1: diving.mp4 ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Action Predicted: diving (Confidence: 1.00)
FINAL PREDICTION: diving

--- Testing Video 2: Biking.mp4 ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Action Predicted: diving (Confidence: 0.75)
FINAL PREDICTION: diving

--- Testing Video 3: basketball.mp4 ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Action Predicted: basketball (Confidence: 1.00)
FINAL PREDICTION: basketball

--- Validation Process Finished ---
